In [1]:
import os
import time
import random

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader

import onnxruntime as ort

import rasterio
from rasterio.windows import Window

import ee
import geemap


### The Preprocessing Gauntlet

cannot just feed raw SAR into a standard convolutional network and expect results.
SAR data is fundamentally different from optical imagery; it is plagued by speckle noise and terrain distortions.
Pull overlapping Sentinel-1 (SAR) Ground Range Detected (GRD) data and Sentinel-2 (Optical) multispectral data for the same geographical target within a tight temporal window (ideally under 48 hours apart).Calibrate the SAR (Sentinel-1): SAR backscatter needs to be converted into a meaningful physical quantity.

Apply precise orbit files, perform thermal noise removal, radiometric calibration to Sigma-nought ($\sigma^0$), and terrain correction using a Digital Elevation Model (DEM) to fix layover and shadow effects.Filter the Speckle: Apply a Lee or Refined Lee filter to the SAR data. If you skip this, your model will mistake SAR noise for structural features.Clean the Optical (Sentinel-2): Apply atmospheric correction (Sen2Cor) to convert Top-Of-Atmosphere (TOA) reflectance to Bottom-Of-Atmosphere (BOA) reflectance.

In [ ]:

#intiliase conncetion
ee.Authenticate()
ee.Initialize()

#Define the target vwector (Bangalore region)
roi=ee.Geometry.Rectangle([77.5,12.9,77.6,13.0])
start_date,end_date='2025-09-01','2025-10-31'


#1.SAR Pipeline(Senital-1)
# Radiometric calibration and terrain correction are handled natively by the GRD product in GEE.
s1_raw=(ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(roi)
        .filterDate(start_date,end_date)
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
        .filter(ee.Filter.eq('instrumentMode','IW'))
        .first().clip(roi))

# 2. Optical Pipeline (Sentinel-2)
# The 'SR_HARMONIZED' collection is already atmospherically corrected (Sen2Cor) to Bottom-Of-Atmosphere (BOA).
s2_optical=(ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi)
            .filterDate(start_date,end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',40))
            .first().clip(roi)
            .select(['B4','B3','B2','B8']))#red,blue,green,NIR

print("Done")

In [ ]:
## Speckle Filter: Apply a 5x5 focal median (approximate Lee filter for structural preservation)
# Select only the two radar bands before applying the speckle filter
s1_filtered = s1_raw.select(['VV', 'VH']).focal_median(radius=5, units='pixels')
#Export local Master GeoTIFF at exact 10m scale
geemap.ee_export_image(s1_filtered, filename='SAR_master.tif', scale=10, region=roi)
geemap.ee_export_image(s2_optical, filename='Optical_master.tif', scale=10, region=roi)
                       

### Coregistration

Coregistration (The Make-or-Break Step) 
Sentinel-1 pixels and Sentinel-2 pixels do not align perfectly, your fusion model will learn absolute garbage.Resample to a Common Grid: Sentinel-1 operates at a 10m resolution, while Sentinel-2 bands vary (10m, 20m, 60m). Resample all relevant bands to a unified 10m spatial resolution.Pixel-Level Alignment: Georeference both datasets to the exact same Coordinate Reference System (CRS).

In [ ]:
### The GEE export locked the 10m resolution. 
### Now we enforce pixel-level spatial alignment locally and slice the massive matrices into overlapping 256x256 tensors.

In [ ]:
os.makedirs('dataset/tensors',exist_ok=True)
print("Done")

In [ ]:
def extract_coregistered_patches(sar_path, opt_path, patch_size=256):
    with rasterio.open(sar_path) as src_sar, rasterio.open(opt_path) as src_opt:
        
        # Enforce exact spatial grid match
        h = min(src_sar.shape[0], src_opt.shape[0])
        w = min(src_sar.shape[1], src_opt.shape[1])
        
        patch_count = 0
        for i in range(0, h - patch_size + 1, patch_size):
            for j in range(0, w - patch_size + 1, patch_size):
                window = Window(j, i, patch_size, patch_size)

                sar_patch = src_sar.read(window=window) # [VV, VH]
                opt_patch = src_opt.read(window=window) # [R, G, B, NIR]

                # Drop edge anomalies (dead sensors)
                if np.any(sar_patch == 0) or np.any(opt_patch == 0):
                    continue

                # Coregistration: Stack into a single 6-channel tensor [VV, VH, R, G, B, NIR]
                fused_tensor = np.concatenate((sar_patch, opt_patch), axis=0)
                np.save(f'dataset/tensors/patch_{patch_count}.npy', fused_tensor)
                patch_count += 1
                
    print(f"{patch_count} coregistered 6-channel tensors generated.")

extract_coregistered_patches('SAR_master.tif', 'Optical_master.tif')

### Architecture Engineering 

 Architecture Engineering

Build a Two-Stream Architecture: Design a dual-encoder network (like a modified U-Net or a Vision Transformer variant).

Stream A (The SAR Encoder): Processes the [VV, VH] bands. It learns to extract structural and topological features.

Stream B (The Optical Encoder): Processes the [RGB, NIR] bands. It learns to extract spectral and vegetation indices.

Feature-Level (Late) Fusion: Concatenate or use attention mechanisms to fuse the latent representations from both encoders at various depths before passing them into a unified decoder. This allows the network to learn the non-linear relationships between radar texture and optical color.

In [ ]:

#### We explicitly isolate radar physics from optical physics using two independent convolutional encoders before fusing their latent representations.

In [ ]:

class ResidualEncoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
    def forward(self, x):
        return self.conv(x)

class DualStreamOptoSAR(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        # Stream A: SAR Physics (2 Channels)
        self.sar_stem = nn.Sequential(
            ResidualEncoderBlock(2, 32),
            ResidualEncoderBlock(32, 64)
        )
        
        # Stream B: Optical Physics (4 Channels)
        self.opt_stem = nn.Sequential(
            ResidualEncoderBlock(4, 32),
            ResidualEncoderBlock(32, 64)
        )

        # Late Feature Fusion Decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=1)
        )

    # CRITICAL: This must be indented exactly here, inside the class
    def forward(self, sar, opt):
        sar_latent = self.sar_stem(sar)
        opt_latent = self.opt_stem(opt)
        
        # Concatenate latent maps along the channel dimension
        fused = torch.cat((sar_latent, opt_latent), dim=1) 
        return self.decoder(fused)

print("Architecture redefined and loaded into memory.")

In [ ]:
def verify_architecture():
    print("Initiating Architecture Verification Protocol...")
    
    # 1. Hardware Targeting
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Compiling on: {device}")
    
    # 2. Deploy the architecture
    # Ensure DualStreamOptoSAR is already defined in the cell above
    model = DualStreamOptoSAR(num_classes=5).to(device)
    
    # 3. Fabricate synthetic payload tensors
    # Simulating a Batch Size of 2
    # SAR expects: 2 channels (VV, VH) at 256x256
    # Optical expects: 4 channels (R, G, B, NIR) at 256x256
    dummy_sar = torch.randn(2, 2, 256, 256).to(device)
    dummy_opt = torch.randn(2, 4, 256, 256).to(device)
    
    print(f"Injected SAR Tensor: {dummy_sar.shape}")
    print(f"Injected Optical Tensor: {dummy_opt.shape}")
    
    # 4. Execute the Forward Pass
    try:
        output_map = model(dummy_sar, dummy_opt)
        
        # We expect an output of [2, 5, 256, 256] 
        # (2 images, 5 terrain classes, 256x256 resolution)
        print(f"\nNETWORK STATUS: GREEN.")
        print(f"Generated Output Tensor: {output_map.shape}")
        print("Mathematical graph is sound. Proceed to Phase 4.")
        
    except Exception as e:
        print(f"\nCRITICAL FAILURE in Forward Pass:")
        print(e)

# Fire the verification engine
verify_architecture()

### Training for Cloud Resilience

This is where train the model to actually solve the problem of obstructed views. force the network to rely on the SAR stream when the optical stream goes blind.

Simulate Obstruction: During the data augmentation phase of training, intentionally mask out random regions of the optical patches with simulated cloud cover or Gaussian noise.

Force the Dependency: Because the optical data is corrupted, the loss function will penalize the model heavily unless it learns to extract the necessary terrain features solely from the uncorrupted SAR stream.

Loss Optimization: Use a combination of Cross-Entropy Loss and Dice Loss to ensure the model focuses on minority classes and sharp boundaries in the terrain mapping.

In [ ]:
### We engineer an aggressive PyTorch dataset that intentionally sabotages the optical feed during training. 
##We combine Cross-Entropy (for pixel accuracy) and Dice Loss (for sharp geometric boundaries).

In [ ]:


# --- 1. The Dice Loss Function ---
# Enforces sharp geometric boundaries on terrain features
class DiceLoss(nn.Module):
    def forward(self, inputs, targets, smooth=1):
        inputs = F.softmax(inputs, dim=1)
        # Convert integer mask to one-hot encoding
        targets_one_hot = F.one_hot(targets, num_classes=inputs.shape[1]).permute(0, 3, 1, 2).float()
        intersection = (inputs * targets_one_hot).sum(dim=(2, 3))
        dice = (2. * intersection + smooth) / (inputs.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3)) + smooth)
        return 1 - dice.mean()

# --- 2. The Cloud Sabotage Dataset ---
class CombatDataset(Dataset):
    def __init__(self, tensor_dir, mask_dir, is_training=True):
        self.tensor_dir = tensor_dir
        self.mask_dir = mask_dir
        self.is_training = is_training
        # Sort files to ensure array alignment
        self.files = sorted([f for f in os.listdir(tensor_dir) if f.endswith('.npy')])
        os.makedirs(mask_dir, exist_ok=True) # Ensure mask directory exists

    def __len__(self): 
        return len(self.files)

    def __getitem__(self, idx):
        # 1. Load the 6-channel fused tensor from Phase 2
        data = torch.from_numpy(np.load(os.path.join(self.tensor_dir, self.files[idx]))).float()
        
        # 2. Split back into SAR (first 2 channels) and Optical (last 4 channels)
        sar = data[:2, :, :] 
        opt = data[2:, :, :] 
        
        # 3. Min-Max Normalization
        sar = sar / (torch.max(sar) + 1e-8)
        opt = opt / (torch.max(opt) + 1e-8)

        # --- THE SABOTAGE PROTOCOL ---
        if self.is_training and random.random() > 0.4:
            # Total Cloud Cover: Blind a random 64x64 pixel zone in the optical feed
            x, y = random.randint(0, 192), random.randint(0, 192)
            opt[:, y:y+64, x:x+64] = 1.0 # 1.0 = Max reflectance (Whiteout)
            
            # Atmospheric Interference: Inject Gaussian sensor noise
            opt += torch.randn_like(opt) * 0.1
            opt = torch.clamp(opt, 0.0, 1.0) # Keep values within valid physical bounds

        # 4. Load Ground Truth Mask
        mask_path = os.path.join(self.mask_dir, self.files[idx])
        if os.path.exists(mask_path):
            mask = torch.from_numpy(np.load(mask_path)).long().squeeze()
        else:
            # Failsafe: If no mask exists yet, generate a dummy mask to prevent the engine from crashing
            mask = torch.zeros((256, 256), dtype=torch.long) 

        return sar, opt, mask

# --- 3. The Optimization Engine ---
def train_resilient_model():
    print("Phase 4: Optimization Engine Engaged.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    
    # Deploy Architecture (DualStreamOptoSAR must already be in memory from Phase 3)
    model = DualStreamOptoSAR(num_classes=5).to(device)
    
    # Loss Functions
    ce_loss = nn.CrossEntropyLoss()
    dice_loss = DiceLoss()
    
    # Optimizer (AdamW with weight decay prevents overfitting)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    # Initialize Data Pipeline
    dataset = CombatDataset(tensor_dir='dataset/tensors', mask_dir='dataset/masks')
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True) # Batch size 2 protects RTX 3050 VRAM

    epochs = 20
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for sar, opt, mask in dataloader:
            sar, opt, mask = sar.to(device), opt.to(device), mask.to(device)
            
            # Zero Gradients
            optimizer.zero_grad()
            
            # Forward Pass
            predictions = model(sar, opt)
            
            # Calculate Combined Loss (Accuracy + Structure)
            loss = ce_loss(predictions, mask) + dice_loss(predictions, mask)
            
            # Backward Pass & Optimize
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / max(len(dataloader), 1)
        print(f"Epoch [{epoch+1}/{epochs}] | Combined Loss: {avg_loss:.4f}")
            
    # Secure the finalized weights
    torch.save(model.state_dict(), "optosar_resilient_weights.pth")
    print("Phase 4 Complete. Cloud-resilient weights secured locally.")

# Execute the training pipeline
train_resilient_model()

### Deployment & Edge Optimization


A heavy model sitting on a server is useless for autonomous navigation or real-time satellite edge computing.

Prune and Quantize: Strip out redundant weights. Quantize your model from FP32 to INT8.

Compile for the Hardware: Export the weights to ONNX format and optimize the computation graph using NVIDIA TensorRT.

Execute this pipeline, and you transition from analyzing clear-sky pictures to engineering a perception system that operates relentlessly, regardless of the atmosphere. Get the datasets, align the grids, and start building the dual-encoder.

In [ ]:


def compile_edge_engine(weights_path="optosar_resilient_weights.pth"):
    print("Phase 5: Edge Compilation Protocol Engaged.")
    
    # 1. Hardware Targeting (Exporting on CPU is safer for ONNX tracing)
    device = torch.device("cpu")
    
    # 2. Deploy Architecture & Load Weights
    # Ensure DualStreamOptoSAR is in memory from Phase 3
    model = DualStreamOptoSAR(num_classes=5).to(device)
    
    if not os.path.exists(weights_path):
        print(f"CRITICAL: {weights_path} not found. Did you complete Phase 4?")
        return
        
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval() # Lock the graph for inference
    print("Heavy model loaded into memory.")

    # --- PROTOCOL A: NETWORK PRUNING ---
    print("Executing L1 Unstructured Pruning...")
    pruned_layers = 0
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            # Strip the bottom 25% of the weakest weights in every convolutional layer
            prune.l1_unstructured(module, name='weight', amount=0.25)
            prune.remove(module, 'weight') # Commit the pruning permanently
            pruned_layers += 1
            
    print(f"Severed dead weights across {pruned_layers} layers.")

    # --- PROTOCOL B: QUANTIZATION ---
    print("Compressing precision (Quantization)...")
    # Compress standard Linear operations to 8-bit integers to save memory
    quantized_model = torch.quantization.quantize_dynamic(
        model, {nn.Linear}, dtype=torch.qint8
    )

    # --- PROTOCOL C: ONNX EXPORT ---
    print("Tracing computation graph for ONNX export...")
    
    # Fabricate a Batch Size 1 dummy payload to trace the graph
    dummy_sar = torch.randn(1, 2, 256, 256).to(device)
    dummy_opt = torch.randn(1, 4, 256, 256).to(device)

    onnx_filename = "OptoSAR_Edge_Engine.onnx"
    
    torch.onnx.export(
        quantized_model,                                
        (dummy_sar, dummy_opt),              
        onnx_filename,                      
        export_params=True,                  # Embed the trained weights inside the file
        opset_version=12,                    # Standard version for maximum TensorRT compatibility
        do_constant_folding=True,            # Pre-calculate constant nodes to speed up edge inference
        input_names=['SAR_Sensor_Input', 'Optical_Sensor_Input'], 
        output_names=['Terrain_Intelligence_Output'],    
        dynamic_axes={                       # Allow the edge device to use variable batch sizes
            'SAR_Sensor_Input': {0: 'batch_size'},
            'Optical_Sensor_Input': {0: 'batch_size'},
            'Terrain_Intelligence_Output': {0: 'batch_size'}
        }
    )

    print(f"\nCOMPILATION COMPLETE.")
    print(f"Deployment Payload Secured: {onnx_filename}")
    print("Your model is now framework-agnostic. Proceed to TensorRT ingestion on target silicon.")

# Fire the compiler
compile_edge_engine()

In [ ]:


def deploy_onnx_visualizer(patch_index=0):
    print("Engaging ONNX Runtime Inference Engine...")
    onnx_model_path = "OptoSAR_Edge_Engine.onnx"
    
    # 1. Verify Payload
    if not os.path.exists(onnx_model_path):
        print("CRITICAL: ONNX payload missing. Did you complete Phase 5?")
        return

    # 2. Load the Framework-Agnostic Engine
    # We execute on the CPU provider to prove it runs independently of heavy GPU clusters
    session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
    
    # 3. Load Raw Intelligence (Tensors generated in Phase 2)
    tensor_path = f'dataset/tensors/patch_{patch_index}.npy'
    if not os.path.exists(tensor_path):
        print(f"CRITICAL: Tensor dataset/tensors/patch_{patch_index}.npy not found.")
        return
        
    data = np.load(tensor_path).astype(np.float32)
    sar_raw = data[:2, :, :]
    opt_raw = data[2:, :, :] # The 4 optical channels
    
    # 4. Exact Preprocessing (Min-Max Normalization to match training math)
    sar_norm = sar_raw / (np.max(sar_raw) + 1e-8)
    opt_norm = opt_raw / (np.max(opt_raw) + 1e-8)
    
    # Add the batch dimension required by the graph: [1, Channels, Height, Width]
    sar_input = np.expand_dims(sar_norm, axis=0)
    opt_input = np.expand_dims(opt_norm, axis=0)
    
    # 5. Execute ONNX Inference and Measure Latency
    inputs = {
        'SAR_Sensor_Input': sar_input,
        'Optical_Sensor_Input': opt_input
    }
    
    print("Executing forward pass through ONNX graph...")
    start_time = time.time()
    
    # Output is a list, we want the first element which contains the logits: shape [1, 5, 256, 256]
    logits = session.run(None, inputs)[0] 
    
    inference_time = (time.time() - start_time) * 1000
    print(f"Inference complete. Engine Latency: {inference_time:.2f} ms")
    
    # 6. Post-Processing (Extract the class with the highest probability)
    prediction_map = np.argmax(logits, axis=1).squeeze()
    
    # 7. Visualization Render Protocol
    print("Rendering visual intelligence...")
    cmap = mcolors.ListedColormap(['red', 'blue', 'darkgreen', 'lightgreen', 'gray'])
    bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Panel 1: Optical RGB
    rgb_display = opt_raw[:3, :, :].transpose(1, 2, 0) # Grab Red, Green, Blue
    rgb_display = (rgb_display - np.min(rgb_display)) / (np.max(rgb_display) - np.min(rgb_display) + 1e-8)
    axes[0].imshow(rgb_display)
    axes[0].set_title('Raw Optical Feed (RGB)')
    axes[0].axis('off')

    # Panel 2: SAR Feed (VV Polarization)
    axes[1].imshow(sar_raw[0, :, :], cmap='gray')
    axes[1].set_title('Raw SAR Feed (VV)')
    axes[1].axis('off')

    # Panel 3: ONNX Prediction Output
    axes[2].imshow(prediction_map, cmap=cmap, norm=norm)
    axes[2].set_title('Edge Compiled Terrain Map (ONNX Output)')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Fire the visualizer on the first patch
deploy_onnx_visualizer(patch_index=0)